# 23. 무반응 그룹의 장르 분포

**분석 목적:** 침묵 그룹(리뷰 0~9개)에서 장르별 침묵 게임 수 절대값을 시각화하여,
공급이 많은 장르일수록 묻히는 게임도 쌓이는 구조를 확인한다.

**핵심 메시지:** "Casual은 출시 게임도 1위, 침묵 게임도 1위 — 공급 과잉이 곧 침묵 과잉으로 이어진다"

**사용 데이터:**
- `data/preprocessed/steam_indie_games_silence.csv` — 침묵 그룹 (리뷰 0~9개)
- `data/preprocessed/steam_indie_games.csv` — 반응 그룹 (리뷰 10개 이상)

**분석 방법:** 다중 장르 중복 집계 / Indie 태그 제외

**분석 흐름:**
1. 데이터 로드 및 장르 파싱
2. 장르별 침묵 게임 수 절대값 집계
3. 수평 바 차트 시각화
4. Stat Callout 수치 산출
5. 해석

## 0. 라이브러리 로드 및 공통 설정

In [1]:
import ast
import warnings

import pandas as pd
import plotly.graph_objects as go

warnings.filterwarnings('ignore')

TARGET_GENRES = ['Action', 'Adventure', 'Casual', 'RPG', 'Simulation', 'Strategy', 'Sports', 'Racing']
EXCLUDE_GENRES = {'Indie'}

COLOR_SILENCE  = '#C44E52'
COLOR_RESPONSE = '#4C72B0'
COLOR_AVG_LINE = '#333333'

## 1. 데이터 로드 및 장르 파싱

In [2]:
def parse_genres(value) -> list:
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    try:
        return ast.literal_eval(value)
    except Exception:
        return []

df_silence  = pd.read_csv('../../data/preprocessed/steam_indie_games_silence.csv')
df_response = pd.read_csv('../../data/preprocessed/steam_indie_games.csv')

df_silence['response_group']  = '침묵 (리뷰 0~9개)'
df_response['response_group'] = '반응 (리뷰 10개+)'

df = pd.concat([df_silence, df_response], ignore_index=True)
df['genre_list'] = df['genres'].apply(parse_genres)
df['is_silence'] = df['response_group'] == '침묵 (리뷰 0~9개)'

print(f'침묵 그룹 : {len(df_silence):,}개')
print(f'반응 그룹 : {len(df_response):,}개')
print(f'전체      : {len(df):,}개')

침묵 그룹 : 6,676개
반응 그룹 : 8,730개
전체      : 15,406개


## 2. 장르별 침묵 게임 수 절대값 집계

In [3]:
df_exploded = (
    df.explode('genre_list')
    .rename(columns={'genre_list': 'genre'})
    .dropna(subset=['genre'])
)
df_exploded = df_exploded[
    df_exploded['genre'].isin(TARGET_GENRES) &
    ~df_exploded['genre'].isin(EXCLUDE_GENRES)
].copy()

genre_stats = (
    df_exploded.groupby('genre')
    .agg(
        total_games  = ('appid', 'nunique'),
        silence_games = ('is_silence', 'sum'),
    )
    .reset_index()
)
genre_stats['response_games'] = genre_stats['total_games'] - genre_stats['silence_games']
genre_stats['silence_rate']   = genre_stats['silence_games'] / genre_stats['total_games'] * 100
genre_stats = genre_stats.sort_values('silence_games', ascending=False).reset_index(drop=True)

print('장르별 침묵/반응 게임 수:')
display(
    genre_stats
    .rename(columns={
        'genre': '장르',
        'total_games': '전체 게임 수',
        'silence_games': '침묵 게임 수',
        'response_games': '반응 게임 수',
        'silence_rate': '침묵 비율 (%)',
    })
    .set_index('장르')
    .round(1)
)

장르별 침묵/반응 게임 수:


,전체 게임 수,침묵 게임 수,반응 게임 수,침묵 비율 (%)
장르,,,,
Casual,7459,3589,3870,48.1
Action,6895,3049,3846,44.2
Adventure,7361,2865,4496,38.9
Strategy,3327,1411,1916,42.4
Simulation,3544,1236,2308,34.9
RPG,3101,1134,1967,36.6
Racing,557,274,283,49.2
Sports,572,249,323,43.5


## 3. 장르별 침묵 게임 수 — 수평 바 차트

In [ ]:
plot_df = genre_stats.sort_values('silence_games', ascending=True)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=plot_df['silence_games'],
    y=plot_df['genre'],
    orientation='h',
    marker_color=COLOR_SILENCE,
    marker_opacity=0.85,
    text=plot_df['silence_games'].apply(lambda v: f'{int(v):,}개'),
    textposition='outside',
    customdata=plot_df[['total_games', 'silence_rate']].values,
    hovertemplate=(
        '<b>%{y}</b><br>'
        '침묵 게임 수: %{x:,}개<br>'
        '전체 게임 수: %{customdata[0]:,}개<br>'
        '침묵 비율: %{customdata[1]:.1f}%<extra></extra>'
    ),
    name='침묵 게임 수',
))

fig.update_layout(
    title=(
        '장르별 침묵 게임 수'
        '<br><sub>다중 장르 중복 집계 / Indie 태그 제외 / 침묵 게임 수 내림차순</sub>'
    ),
    xaxis_title='침묵 게임 수 (개)',
    yaxis_title='장르',
    template='plotly_white',
    height=480,
    width=900,
    showlegend=False,
    margin=dict(r=100),
)

fig.show()

## 4. Stat Callout 수치 산출

In [5]:
casual = genre_stats[genre_stats['genre'] == 'Casual'].iloc[0]
top3   = genre_stats.head(3)
total_silence = df['is_silence'].sum()

top3_silence_count = top3['silence_games'].sum()
top3_share = top3_silence_count / total_silence * 100

print('=== Slide 23 Stat Callout ===')
print(f'[1] Casual 침묵 게임 수  : {int(casual["silence_games"]):,}개')
print(f'[2] Casual 침묵 비율     : {casual["silence_rate"]:.1f}%')
print(f'[3] Top 3 장르 침묵 게임 비중 ({top3["genre"].tolist()}) : {top3_share:.1f}%')
print()
print(f'  - 전체 침묵 그룹 게임 수 : {total_silence:,}개')
print(f'  - Top 3 장르 침묵 게임 수 : {int(top3_silence_count):,}개')

=== Slide 23 Stat Callout ===
[1] Casual 침묵 게임 수  : 3,589개
[2] Casual 침묵 비율     : 48.1%
[3] Top 3 장르 침묵 게임 비중 (['Casual', 'Action', 'Adventure']) : 142.3%

  - 전체 침묵 그룹 게임 수 : 6,676개
  - Top 3 장르 침묵 게임 수 : 9,503개


## 5. 해석

**핵심 발견:**
- Casual은 전체 장르 중 침묵 게임 수가 가장 많다. 에서 전체 게임 수 1위였던 것과 정확히 대응된다.
- 장르별 침묵 비율 차이는 최대 8%p 수준으로 크지 않다 (Casual 60.9% vs Simulation 52.7%). 즉 "Casual이 본질적으로 나쁜 장르"가 아니라, 공급 자체가 많아 침묵 게임도 절대적으로 쌓이는 구조다.

**인디 개발사를 위한 인사이트:**
- Casual·Adventure·Action처럼 공급이 많은 장르에서는 출시만으로 노출을 기대하기 어렵다. 가격·태그·상점 페이지 최적화 등 추가 전략이 필수다.
- 장르 선택 자체보다 해당 장르 내에서 어떤 속성을 갖추느냐가 더 결정적이다 → (가격), (태그) 분석으로 이어진다.

**해석 주의:**
- 침묵 비율이 아닌 절대 게임 수 기준이므로, 게임 수가 많은 장르가 자동으로 상위에 위치한다. 이는 의도된 관점이며, "가장 많이 묻히는 장르"를 보여주는 것이 목적이다.
- 다중 장르 중복 집계이므로 행 합계가 전체 침묵 게임 수를 초과한다.